# Step 4 — Query & Reasoning
Mengeksplorasi Knowledge Graph film menggunakan Cypher query.

Query yang dibuat:
1. Film dengan aktor yang sama
2. Film dengan genre + sutradara yang sama
3. Aktor paling produktif
4. Genre terpopuler
5. Sutradara paling produktif
6. Jarak graph antara dua film
7. Film berdasarkan keyword tertentu

In [4]:
import pandas as pd
from neo4j import GraphDatabase

# bolt:// untuk koneksi lokal (bukan neo4j:// yang untuk cluster)
URI      = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "password123"
DATABASE = "flimkg"

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print('Koneksi berhasil!')

def query(cypher, **params):
    with driver.session(database=DATABASE) as session:
        result = session.run(cypher, **params).data()
    if not result:
        print('  [!] Tidak ada hasil ditemukan.')
        return pd.DataFrame()
    return pd.DataFrame(result)

Koneksi berhasil!


---
## Query 1 — Film dengan Aktor yang Sama
Cari film lain yang memiliki minimal 1 aktor yang sama dengan film tertentu.

In [5]:
film_target = "Toy Story"

cypher = """
MATCH (m1:Movie {title: $title})-[:actedBy]->(a:Actor)<-[:actedBy]-(m2:Movie)
WHERE m1 <> m2
WITH m2, collect(a.name) AS aktor_sama, count(a) AS jumlah_aktor_sama
RETURN m2.title AS film, aktor_sama, jumlah_aktor_sama
ORDER BY jumlah_aktor_sama DESC
LIMIT 10
"""

df = query(cypher, title=film_target)
print(f'Film dengan aktor sama seperti "{film_target}":')
df

Film dengan aktor sama seperti "Toy Story":


,film,aktor_sama,jumlah_aktor_sama
0,Toy Story 2,"[Don Rickles, Tim Allen, Tom Hanks]",3
1,Toy Story That Time Forgot,"[Tim Allen, Tom Hanks, Wallace Shawn]",3
2,Toy Story of Terror!,"[Tim Allen, Tom Hanks, Wallace Shawn]",3
3,Partysaurus Rex,"[Tim Allen, Tom Hanks, Wallace Shawn]",3
4,Toy Story 3,"[Tim Allen, Tom Hanks]",2
5,Hawaiian Vacation,"[Tim Allen, Tom Hanks]",2
6,Bikini Beach,[Don Rickles],1
7,Kelly's Heroes,[Don Rickles],1
8,Dirty Work,[Don Rickles],1
9,Mr. Warmth: The Don Rickles Project,[Don Rickles],1


---
## Query 2 — Film dengan Genre + Sutradara yang Sama

In [6]:
film_target = "Toy Story"

cypher = """
MATCH (m1:Movie {title: $title})-[:directedBy]->(d:Director)<-[:directedBy]-(m2:Movie)
WHERE m1 <> m2
MATCH (m1)-[:hasGenre]->(g:Genre)<-[:hasGenre]-(m2)
WITH m2, d, collect(DISTINCT g.name) AS genre_sama
RETURN m2.title         AS film,
       d.name           AS sutradara,
       genre_sama,
       size(genre_sama) AS jumlah_genre_sama
ORDER BY jumlah_genre_sama DESC
LIMIT 10
"""

df = query(cypher, title=film_target)
print(f'Film dengan sutradara + genre sama seperti "{film_target}":')
df

Film dengan sutradara + genre sama seperti "Toy Story":


,film,sutradara,genre_sama,jumlah_genre_sama
0,A Bug's Life,John Lasseter,"[Animation, Comedy, Family]",3
1,Toy Story 2,John Lasseter,"[Animation, Comedy, Family]",3
2,Cars,John Lasseter,"[Animation, Comedy, Family]",3
3,Cars 2,John Lasseter,"[Animation, Comedy, Family]",3
4,Mater and the Ghostlight,John Lasseter,"[Animation, Family]",2
5,Luxo Jr.,John Lasseter,[Animation],1
6,Tin Toy,John Lasseter,[Animation],1
7,Red's Dream,John Lasseter,[Animation],1
8,Knick Knack,John Lasseter,[Animation],1


---
## Query 3 — Aktor Paling Produktif

In [7]:
cypher = """
MATCH (a:Actor)<-[:actedBy]-(m:Movie)
WITH a, count(m) AS jumlah_film
RETURN a.name AS aktor, jumlah_film
ORDER BY jumlah_film DESC
LIMIT 10
"""

df = query(cypher)
print('Top 10 Aktor Paling Produktif:')
df

Top 10 Aktor Paling Produktif:


,aktor,jumlah_film
0,John Wayne,107
1,Jackie Chan,101
2,Christopher Lee,97
3,Gérard Depardieu,96
4,Michael Caine,96
5,Robert De Niro,96
6,Samuel L. Jackson,84
7,Donald Sutherland,81
8,Danny Glover,78
9,Harvey Keitel,77


---
## Query 4 — Genre Terpopuler

In [8]:
cypher = """
MATCH (g:Genre)<-[:hasGenre]-(m:Movie)
WITH g, count(m) AS jumlah_film
RETURN g.name AS genre, jumlah_film
ORDER BY jumlah_film DESC
LIMIT 10
"""

df = query(cypher)
print('Top 10 Genre Terpopuler:')
df

Top 10 Genre Terpopuler:


,genre,jumlah_film
0,Drama,20243
1,Comedy,13176
2,Thriller,7618
3,Romance,6730
4,Action,6590
5,Horror,4670
6,Crime,4304
7,Documentary,3930
8,Adventure,3490
9,Science Fiction,3042


---
## Query 5 — Sutradara Paling Produktif

In [9]:
cypher = """
MATCH (d:Director)<-[:directedBy]-(m:Movie)
WITH d, count(m) AS jumlah_film
RETURN d.name AS sutradara, jumlah_film
ORDER BY jumlah_film DESC
LIMIT 10
"""

df = query(cypher)
print('Top 10 Sutradara Paling Produktif:')
df

Top 10 Sutradara Paling Produktif:


,sutradara,jumlah_film
0,John Ford,68
1,Michael Curtiz,65
2,Werner Herzog,55
3,Alfred Hitchcock,53
4,Georges Méliès,51
5,Jean-Luc Godard,50
6,Woody Allen,49
7,Sidney Lumet,46
8,Charlie Chaplin,44
9,Henry Hathaway,43


---
## Query 6 — Jarak Graph antara Dua Film

In [11]:
film_a = "Toy Story"
film_b = "A Bug's Life"

cypher = """
MATCH (m1:Movie {title: $film_a}),
      (m2:Movie {title: $film_b})
MATCH path = shortestPath((m1)-[*..6]-(m2))
RETURN length(path) AS jarak_graph,
       [n IN nodes(path) |
           CASE
               WHEN n:Movie    THEN n.title
               WHEN n:Actor    THEN '(Aktor) '     + n.name
               WHEN n:Director THEN '(Sutradara) ' + n.name
               WHEN n:Genre    THEN '(Genre) '     + n.name
               WHEN n:Keyword  THEN '(Keyword) '   + n.name
               ELSE n.name
           END
       ] AS jalur
"""

df = query(cypher, film_a=film_a, film_b=film_b)
print(f'Jarak graph antara "{film_a}" dan "{film_b}":')
df

Jarak graph antara "Toy Story" dan "A Bug's Life":


,jarak_graph,jalur
0,2,"[Toy Story, (Sutradara) John Lasseter, A Bug's..."


---
## Query 7 — Film Berdasarkan Keyword Tertentu

In [10]:
keyword_target = "friendship"

cypher = """
MATCH (m:Movie)-[:hasKeyword]->(k:Keyword {name: $keyword})
RETURN m.title AS film, m.release_date AS tanggal_rilis
ORDER BY m.release_date DESC
LIMIT 10
"""

df = query(cypher, keyword=keyword_target)
print(f'Film dengan keyword "{keyword_target}":')
df

Film dengan keyword "friendship":


,film,tanggal_rilis
0,The Trip to Spain,2017-08-31
1,Hampstead,2017-06-23
2,The Last Word,2017-03-03
3,T2 Trainspotting,2017-01-27
4,Heartstone,2016-12-28
5,The Edge of Seventeen,2016-11-18
6,Burn Burn Burn,2016-10-28
7,Die Beautiful,2016-10-27
8,Aanandam,2016-10-21
9,This Beautiful Fantastic,2016-10-20


In [ ]:
driver.close()
print('Koneksi ditutup.')